In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col


In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}

Read bronze table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2_raw")

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

Renaming

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#### Drop if table exists

In [0]:
table_name = "silver.erp_product_category"
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

writing silver Table

In [0]:
(
    df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable("silver.erp_product_category")
)

In [0]:
%sql
select * from silver.erp_product_category